Đây là script chạy phân tích dữ liệu trước khi đi vào xử lý và huấn luyện mô hình. Chi tiết về việc phân tích và nội dung quan sát rút ra đều được ghi chi tiết trong notebook.

**Lưu ý**:
- Script **không bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Thời gian chạy cụ thể: 5-10 phút
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 6 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)

In [ ]:
!git clone https://github.com/CryAndRRich/tagflow.git

In [ ]:
%cd /kaggle/working/tagflow
TAGFLOW_PATH = "/kaggle/working/tagflow"

In [ ]:
!pip install -r requirements.txt

In [4]:
from collections import Counter
import numpy as np
import pandas as pd
import scipy.stats as ss
from scipy.stats import entropy

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
INPUT_ROOT = "YOUR INPUT ROOT PATH"

In [6]:
X = pd.read_csv(f"{INPUT_ROOT}/X_train.csv")
X_val = pd.read_csv(f"{INPUT_ROOT}/X_val.csv")
X_test = pd.read_csv(f"{INPUT_ROOT}/X_test.csv")
Y = pd.read_csv(f"{INPUT_ROOT}/Y_train.csv")

# Phân tích các mã từ X_train

## Các nhóm hành vi
- Nhóm hành động thường xuất hiện đầu
- Nhóm hành động xuất hiện cuối
- Nhóm hành động xuất hiện nhiều, nhưng vị trí không thuộc nhóm những mã xuất hiện đầu và cuối chuỗi 


In [ ]:
features = X.drop(columns=["id"])

long_df = features.melt(var_name="step", value_name="event")
long_df = long_df.dropna()
long_df["step_num"] = long_df["step"].str.extract(r'(\d+)').astype(int)

event_counts = long_df["event"].value_counts()
print("Số giá trị mã:", long_df["event"].nunique())

first_step = long_df[long_df["step_num"] == 1]["event"].value_counts()
entry_ratio = (first_step / event_counts).fillna(0)

last_steps = features.apply(
    lambda row: row.last_valid_index(), axis=1
)

last_events = []
for idx, col in enumerate(last_steps):
    if col is not None:
        last_events.append(features.iloc[idx][col])

last_events = pd.Series(last_events)

terminal_counts = last_events.value_counts()
terminal_ratio = (terminal_counts / event_counts).fillna(0)


summary = pd.DataFrame({
    "total_count": event_counts,
    "entry_ratio": entry_ratio,
    "terminal_ratio": terminal_ratio
}).fillna(0)

print(summary.sort_values("total_count", ascending=False).head(20))

- Phân tích tần suất xuất hiện của các mã, ta thấy 3 mã phổ biến nhất là mã 105, 102 và 103. Tuy nhiên, các mã này có xác suất xuất hiện ở đầu/cuối chuỗi thấp, nên những mã này sẽ là những mã đại diện cho các hành động phổ biến liên quan đến việc lướt ứng dụng/trình duyệt (browsing), xem sản phẩm, thêm vào giỏ hàng.

In [ ]:
sequences = []
for index, row in X.drop("id", axis=1).iterrows():
    seq = [int(x) for x in row.values if not np.isnan(x)]
    if seq:
        sequences.append(seq)

count_A_105_A = 0 
count_A_105_B = 0

for seq in sequences:
    for i in range(1, len(seq) - 1):
        if seq[i] == 105:
            if seq[i-1] == seq[i+1]:
                count_A_105_A += 1
            else:
                count_A_105_B += 1

print(f"Số lần người dùng lặp lại hành động: {count_A_105_A}")
print(f"Số lần người dùng chuyển hướng mới: {count_A_105_B}")
print(f"Tỷ lệ chuyển hướng: {(count_A_105_B / (count_A_105_A + count_A_105_B))*100:.2f}%")

- Hơn 99% thời gian, mã 105 đóng vai trò là cầu nối giữa hai hành động khác biệt => 105 là nút điều hướng trung gian (như nút "Trở về home", "Xem sản phẩm liên quan", "Sang trang"). Các mã 102 và 103 cũng có pattern y hệt 105 nhưng với tần suất thấp hơn. Hành vi kẹp giữa các thao tác này phản ánh một người dùng đang trong giai đoạn browsing tìm hiểu các sản phẩm kỹ càng trước khi ra quyết định ở nhóm mã cuối.

In [ ]:
first_events = [seq[0] for seq in sequences]
last_events = [seq[-1] for seq in sequences]

print("Các sự kiện mở đầu:")
print(pd.Series(first_events).value_counts().head(5))

print("\nCác sự kiện kết thúc:")
print(pd.Series(last_events).value_counts().head(5))

- Nhóm hành động thường xuất hiện đầu: Các mã 760, 685, 836 thường đứng ở vị trí đầu tiên của chuỗi hành vi. Đây là các hành động "Mở ứng dụng" hoặc "Truy cập trang chủ". Việc có 2-3 mã cùng xuất hiện nhiều ở đầu chuỗi (Với 2 mã 760 cà 685 phổ biến hơn hẳn) cho thấy có thể đây là các mã đại diện cho việc truy cập từ nhiều nguồn khác nhau (Truy cập trực tiếp qua app, click từ link quảng cáo, ...).

- Nhóm hành động xuất hiện cuối: 1027, 995, 1080, 975 thường là các hành động cuối cùng chốt lại chuỗi. Đây có thể là các hành vi liên quan đến thanh toán thành công, huỷ đơn hàng, thoát ra khỏi application.

In [ ]:
view_codes = {105, 102, 103}
id_candidates = []
for s in sequences:
    for i in range(1, len(s) - 1):
        if s[i-1] in view_codes and s[i+1] in view_codes:
            id_candidates.append(s[i])
id_gr = [a for a, c in Counter(id_candidates).most_common(10)]

print("Nhóm ID của sản phẩm/ngành hàng:", id_gr[:10])

- Nhóm các mã có thể là ID của sản phẩm hoặc ID của danh mục sản phẩm: Các mã này thường có giá trị gần 1000 đi kèm (kẹp giữa) các mã nhóm điều hướng trung gian/xem sản phẩm (102,103,105). Ví dụ: Người dùng thực hiện thao tác xem, hệ thống ghi nhận họ đang xem sản phẩm 929, sau đó họ lại tiếp tục thao tác xem.

In [ ]:
all_actions = [a for s in sequences for a in s]
counts = Counter(all_actions)
transitions = []
for s in sequences:
    for i in range(len(s) - 1):
        transitions.append((s[i], s[i+1]))

trans_df = pd.DataFrame(transitions, columns=['From', 'To'])

top_20 = [a for a, c in counts.most_common(20)]
matrix = pd.crosstab(trans_df['From'], trans_df['To'], normalize='index')
matrix_filtered = matrix.loc[matrix.index.isin(top_20), matrix.columns.isin(top_20)]

plt.figure(figsize=(12, 10))
sns.heatmap(matrix_filtered, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Ma trận xác suất chuyển đổi hành vi")
plt.xlabel("Hành động tiếp theo (B)")
plt.ylabel("Hành động hiện tại (A)")
plt.show()

- Từ transition matrix, ta có thể thấy các cặp chuyển đổi như 10477 -> 697 và 2207 -> 8615 có xác suất xấp xỉ 100%. Điều này chứng tỏ đây là các bước kỹ thuật bắt buộc của hệ thống, như chuyển trang tự động hoặc load dữ liệu sau một click

- Sau bước đệm chung (mã 8615), người dùng có xu hướng tỏa ra nhiều hướng khác nhau. Chỉ khoảng 32% chuyển sang mã 105 (giả định là mã thể hiện việc người dùng sẽ lướt tiếp), số còn lại phân tán sang các thao tác khác, cho thấy đây là giai đoạn người dùng bắt đầu đưa ra quyết định cá nhân thay vì đi theo luồng mặc định.

In [ ]:
rare_threshold = 10
rare_codes = {code: count for code, count in counts.items() if count < rare_threshold}

print(f"Số lượng mã hiếm: {len(rare_codes)}")
print(f"Một số mã hiếm và số lần xuất hiện: {list(rare_codes.items())[:5]}")

In [ ]:
# Các session rất ngắn
short_seqs = [seq for seq in sequences if len(seq) == 3]
count_103_middle = sum(1 for seq in short_seqs if seq[1] == 103)

print(f"Có {len(short_seqs)} chuỗi chỉ dài đúng 3 hành động.")
print(f"Trong đó, có {count_103_middle} chuỗi có mã 103 nằm ngay chính giữa")
print("Ví dụ 5 chuỗi đầu tiên:", short_seqs[:5])


- Đây là một nhóm người dùng (hoặc loại giao dịch) mang đặc thù riêng biệt. Rất có thể hành vi Mã A -> 103 -> Mã B đại diện cho một thao tác thanh toán tự động định kỳ, mua hàng nhanh (1-click purchase), hoặc hành vi của tài khoản mới tạo.

In [ ]:
bigrams = []
trigrams = []
for s in sequences:
    for i in range(len(s) - 1):
        bigrams.append(tuple(s[i:i+2]))
    for i in range(len(s) - 2):
        trigrams.append(tuple(s[i:i+3]))

top_bi = Counter(bigrams).most_common(10)
top_tri = Counter(trigrams).most_common(10)

print(" Top 10 pattern 2 hành động liên tiếp:")
for p, c in top_bi: print(f"Pattern {p}: {c} lần")

print("\nTop 10 pattern 3 hành động liên tiếp:")
for p, c in top_tri: print(f"Pattern {p}: {c} lần")

Việc soi chiếu các hành động cụm 2 và 3 hành động phổ biến nhất cho thấy các flow lý tưởng của ứng dụng:

- Hai cụm hành vi phổ biến nhất là (685 -> 2207 -> 8615) và (760 -> 1593 -> 8615). Mặc dù bắt đầu từ các điểm vào khác nhau (685 và 760), tất cả đều hội tụ về mã 8615 trước khi người dùng thực hiện các hành vi mua sắm thực sự.
- Mã 8615 là ngưỡng khám phá của ứng dụng. Trước 8615, khách hàng đang ở trạng thái vào app, load trang. Sau 8615, họ mới thực sự bước vào trạng thái tâm lý mua sắm (Xuất hiện mã 105 ngay sau 8615).
- Nếu muốn triển khai các chương trình marketing hoặc hệ thống gợi ý cho các sản phẩm, thời điểm ngay sau mã hành động 8615 sẽ là thời điểm vàng vì đây là lúc ý định bắt đầu hình thành rõ nét nhất

# Phân tích các mã từ Y_Train

In [ ]:
attrs = ["attr_1", "attr_2", "attr_3", "attr_4", "attr_5", "attr_6"]

desc_stats = Y[attrs].agg(["min", "max", "mean", "nunique"]).T
print("Thống kê mô tả 6 thuộc tính:")
print(desc_stats)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, attr in enumerate(attrs):
    top_vals = Y[attr].value_counts().head(20).reset_index()
    top_vals.columns = [attr, "count"]
    
    sns.barplot(data=top_vals, x=attr, y="count", ax=axes[i], 
                hue=attr, palette="YlGnBu_r", legend=False)
    
    axes[i].set_title(f"Phân phối top 20 của {attr}", fontsize=14)
    axes[i].set_xlabel("Giá trị mã")
    axes[i].set_ylabel("Số lượng")

plt.tight_layout()
plt.show()

In [ ]:
# Kiểm định giả thuyết: Các attribute 1-2,4-5 là tháng-ngày
def check_date_validity(df, month_col, day_col):
    invalid_mask = (
        ((df[month_col] == 2) & (df[day_col] > 29)) |
        ((df[month_col].isin([4, 6, 9, 11])) & (df[day_col] > 30))
    )
    return invalid_mask.sum()

errors_1_2 = check_date_validity(Y, "attr_1", "attr_2")
errors_4_5 = check_date_validity(Y, "attr_4", "attr_5")

print(f"\nSố lỗi logic nếu (attr_1, attr_2) là Tháng - Ngày: {errors_1_2}")
print(f"Số lỗi logic nếu (attr_4, attr_5) là Tháng - Ngày: {errors_4_5}")

- Khi thử ghép attr_1 (tháng) và attr_2 (ngày), hệ thống phát hiện 348 trường hợp lỗi logic (ví dụ: xuất hiện ngày 30, 31 trong tháng 2). Tương tự với cặp attr_4 và attr_5 có 398 lỗi logic. Vì vậy, ta có thể đưa ra kết luận rằng: Mặc dù dải giá trị là 1-12 và 1-31, đây không phải là ngày tháng mua hàng.

-  Việc các con số trùng khớp với dải ngày tháng có thể là một sự trùng hợp khi mã hóa hoặc đại diện cho một phân loại có quy mô tương đương.

In [ ]:
def calc_entropy(labels):
    prob = labels.value_counts(normalize=True)
    return entropy(prob)

entropy_series = pd.Series({attr: calc_entropy(Y[attr]) for attr in attrs})
print("\nChỉ số Entropy:")
print(entropy_series.sort_values())

Dữ liệu chia thành 3 cặp có tính chất tương đồng hoàn hảo về số lượng giá trị duy nhất:
- Cặp (attr_1, attr_4): Cùng có 12 giá trị. Entropy thấp nhất (~1.73). Đây là nhóm có độ tập trung cao nhất.
- Cặp (attr_2, attr_5): Cùng có 31 giá trị. Entropy trung bình (~3.04).
- Cặp (attr_3, attr_6): Cùng có 99 giá trị. Entropy cao nhất (~4.59). Đây là nhóm đa dạng nhất.

In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    return np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))

corr_matrix = pd.DataFrame(index=attrs, columns=attrs)
for a1 in attrs:
    for a2 in attrs:
        corr_matrix.loc[a1, a2] = cramers_v(Y[a1], Y[a2])

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.astype(float), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Ma trận tương quan Cramer's V giữa các thuộc tính")
plt.show()

Giả thuyết các thuộc tính:
1.  Ngành hàng lớn: attr_1 và attr_4 
- Chỉ có 12 giá trị duy nhất. Đây là các ngành hàng lớn nhất trong hệ thống (Ví dụ: Điện tử, Gia dụng, Thời trang, Mẹ & Bé...).
- Đây là chỉ số quan trọng nhất để phân bổ ngân sách nhập hàng tổng thể. Nếu mô hình dự báo một sự tăng trưởng ở mã ngành hàng số 3,11,12, doanh nghiệp cần chuẩn bị kế hoạch nhập hàng quy mô lớn cho toàn bộ nhóm sản phẩm thuộc ngành này.

2. Danh mục sản phẩm: attr_2 và attr_5.
- Có 31 giá trị duy nhất. Đặc biệt, mã số 1 chiếm tỷ trọng áp đảo.
- Mã số 1 có thể là các mặt hàng thiết yếu/chủ lực (FMCG). Đây là những sản phẩm có vòng đời ngắn, tốc độ tiêu thụ cực nhanh và nhu cầu luôn ổn định. Các mã từ 2-31 đại diện cho các nhóm hàng ngách, hàng theo mùa hoặc hàng có giá trị trung bình.
- Attribute này sẽ giúp doanh nghiệp xác định vị trí ưu tiên trong kho. Nhóm hàng mã 1 cần được lưu trữ tại khu vực dễ tiếp cận nhất để tối ưu tốc độ đóng gói, vì đây là nhóm tạo ra lưu lượng đơn hàng lớn nhất.

3. Thương hiệu: attr_3 và attr_6.
- Có 99 giá trị duy nhất, độ biến động cao nhất.
- Đây có thể là các thương hiệu cụ thể hoặc các nhóm mã hàng chi tiết.
- Là cơ sở để tối ưu tồn kho an toàn. Với những mã thuộc cấp độ này có tần suất cao: Doanh nghiệp nên duy trì tồn kho sẵn có.

6 attribute dự đoán có thể được phân làm 2 nhóm, nhóm 1 bao gồm attribute 1-2-3 là ngành hàng - danh mục - thương hiệu của sản phẩm đầu tiên người dùng mua, và nhóm 2 bao gồm attribute 4-5-6 là ngành hàng - danh mục - thương hiệu của sản phẩm thứ hai người dùng mua.

In [ ]:
total = len(Y)
exact_match = (Y["attr_1"] == Y["attr_4"]) & \
              (Y["attr_2"] == Y["attr_5"]) & \
              (Y["attr_3"] == Y["attr_6"])

print(f"Tỷ lệ trùng khớp hoàn toàn (1-2-3 == 4-5-6): {exact_match.mean() * 100:.2f}%")

In [ ]:
diff_1_4 = Y["attr_1"] - Y["attr_4"]
diff_2_5 = Y["attr_2"] - Y["attr_5"]

print("Thống kê độ lệch attr_1 - attr_4")
stats_1_4 = (diff_1_4.value_counts(normalize=True).head(5) * 100).round(2)
print(stats_1_4.astype(str) + "%")

print("\nThống kê độ lệch attr_2 - attr_5")
stats_2_5 = (diff_2_5.value_counts(normalize=True).head(5) * 100).round(2)
print(stats_2_5.astype(str) + "%")

hist_colors = sns.color_palette("YlGnBu", 10)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(diff_1_4, bins=50, ax=axes[0], color=hist_colors[3])
axes[0].axvline(0, color="red", linestyle="--", label="Độ lệch = 0 (Trùng khớp)")
axes[0].set_title("Phân phối độ lệch (attr_1 - attr_4)")
axes[0].set_xlabel("Giá trị chênh lệch")
axes[0].legend()

sns.histplot(diff_2_5, bins=50, ax=axes[1], color=hist_colors[7])
axes[1].axvline(0, color="red", linestyle="--", label="Độ lệch = 0 (Trùng khớp)")
axes[1].set_title("Phân phối độ lệch (attr_2 - attr_5)")
axes[1].set_xlabel("Giá trị chênh lệch")
axes[1].legend()

plt.tight_layout()
plt.show()

- Phân phối độ lệch của cả hai cặp thuộc tính (attr_1 - attr_4) và (attr_2 - attr_5) đều tập trung thành một đỉnh tại giá trị 0, kết hợp với tỷ lệ trùng khớp đã đo lường trước đó (25.13%, 27.82%) khẳng định rằng: Mối tương quan mạnh giữa các thuộc tính này bản chất là do khách hàng giữ nguyên danh mục sản phẩm giữa 2 lần tương tác liên tiếp.
- Người dùng có xu hướng khoanh vùng mục tiêu rất rõ ràng trong một session. Khi họ đã xem/tương tác với sản phẩm A, họ có xác suất rất cao (hơn 1/4 số trường hợp) sẽ tiếp tục tìm kiếm và xem xét sản phẩm B thuộc cùng ngành hàng lớn (attr_1/4) và cùng danh mục trung (attr_2/5).
- Vì tỷ lệ trùng khớp hoàn toàn cả 3 attribute (1-2-3 == 4-5-6) chỉ có 0.05%, chứng tỏ họ không tương tác lại với chính sản phẩm đó, mà đang tìm kiếm các sản phẩm thay thế hoặc biến thể trong cùng một tệp phân loại.

Với các đặc điểm hành vi trên, doanh nghiệp có thể tối ưu hệ thống như sau:
- Về thuật toán recommendation: Có thể điều chỉnh trọng số, ưu tiên hiển thị mạnh mẽ các mục tiêu có cùng attr_1 và attr_2 với sản phẩm mà khách hàng vừa tương tác gần nhất.
- Về trải nghiệm người dùng: Tại trang chi tiết sản phẩm, việc ưu tiên block hiển thị sản phẩm tương tự (tập trung vào thay thế/so sánh) sẽ mang lại conversion rate vượt trội hơn so với block "Có thể bạn cũng thích" (tập trung vào bán chéo/cross-sell khác ngành hàng).

# Phân tích phân phối input từ tập train, val, test

In [ ]:
train_len = X.drop("id", axis=1).count(axis=1)
val_len = X_val.drop("id", axis=1).count(axis=1)
test_len = X_test.drop("id", axis=1).count(axis=1)


def plot_distribution_shift(train_col, val_col, test_col, feature_name):
    plt.figure(figsize=(10, 6))

    kde_colors = sns.color_palette("YlGnBu",10)
    
    sns.kdeplot(train_col, fill=True, label="Train", color=kde_colors[9], alpha=0.3)
    sns.kdeplot(val_col, fill=True, label="Val", color=kde_colors[2], alpha=0.3)
    sns.kdeplot(test_col, fill=True, label="Test", color=kde_colors[5], alpha=0.3)
    
    plt.title(f"{feature_name}", fontsize=14)
    plt.ylabel("Mật độ phân phối", fontsize=12)
    plt.legend(loc="upper right")
    plt.grid(True, linestyle="--", alpha=0.6)
    
    plt.show()

plot_distribution_shift(train_len, val_len, test_len, "Phân phối độ dài Session")

- Tập train có đỉnh session ở mốc 9 hành động, trong khi tập val và test hội tụ về đỉnh 12 hành động. Điều này cho thấy hành vi lướt app của người dùng trên tập test (có thể ở giai đoạn sau) đã trở nên ổn định, chuẩn hóa và có chủ đích rõ ràng hơn. Họ không lướt quá nhanh (9 hành động) như trước, mà thường chốt hạ mục tiêu trong khoảng 12 hành động.
- Phân phối tập test xuất hiện một đỉnh phụ đột biến ở mốc session rất ngắn (≤ 5 hành động).

Trên thực tế, sự bùng nổ của nhóm session 4 bước này rất có thể đến từ:
- Một chiến dịch marketing/flash sale mới khiến khách hàng click vào app và mua/thoát ngay lập tức.
- Một thay đổi về UI/UX giúp luồng mua hàng được rút ngắn tối đa.

Action: Áp dụng chiến lược oversampling. Các session có độ dài ≤ 5 trong tập train sẽ được nhân bản lên 40 lần. 

Nếu đưa mô hình lên production, bên cạnh mô hình deep learning đã được oversampling, team sẽ thiết kế thêm một cơ chế fallback: Với các khách hàng có session rất ngắn, thay vì dùng model phức tạp để dự đoán, hệ thống sẽ ưu tiên gợi ý các sản phẩm trending hoặc gợi ý lại chính category mà họ vừa click vào để tăng tỷ lệ chốt đơn nhanh.